In [1]:
!pip install python-chess requests tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 16.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=a0d66144937175b33f6bedc7840d533908998db3e3c3a4a7397b79ae8059d6c6
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [2]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/InfoVis/tp-chess/chess_data"

Mounted at /content/drive


In [3]:
import requests
import chess.pgn
import pandas as pd
import io
import os
import time
from tqdm import tqdm

USERNAME = "sugeema"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ChessDataBot/1.0)"
}

In [4]:
def get_pgn_urls(username):
    url = f"https://api.chess.com/pub/player/{username}/games/archives"
    res = requests.get(url, headers=HEADERS)
    if res.status_code != 200:
        print("❌ Error al obtener archivos:", res.status_code)
        return []
    return res.json()["archives"]


def download_pgns(username, max_archives=66):
    archives = get_pgn_urls(username)
    if not archives:
        print("⚠️ No se encontraron archivos")
        return ""

    print(f"📦 Total archivos disponibles: {len(archives)}")
    print(f"⬇️ Descargando últimos {max_archives}...")

    pgn_texts = []
    for url in archives[-max_archives:]:
        print(f"Descargando: {url}")
        res = requests.get(url + "/pgn", headers=HEADERS)
        if res.status_code == 200:
            pgn_texts.append(res.text)
        else:
            print("❌ Error PGN:", res.status_code)
        time.sleep(1)

    print("✅ Descarga completa")
    return "\n\n".join(pgn_texts)


pgn_data = download_pgns(USERNAME, max_archives=66)

📦 Total archivos disponibles: 66
⬇️ Descargando últimos 66...
Descargando: https://api.chess.com/pub/player/sugeema/games/2020/11
Descargando: https://api.chess.com/pub/player/sugeema/games/2020/12
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/01
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/02
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/03
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/04
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/05
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/06
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/07
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/08
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/09
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/10
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/11
Descargando: https://api.chess.com/pub/player/sugeema/

In [5]:
pgn_io = io.StringIO(pgn_data)
games = []
while True:
    g = chess.pgn.read_game(pgn_io)
    if g is None:
        break
    games.append(g)
print(f"✅ {len(games)} partidas cargadas")

✅ 453 partidas cargadas


In [10]:
import requests
import pandas as pd

eco_dfs = []
for letter in ["a", "b", "c", "d", "e"]:
    url = f"https://raw.githubusercontent.com/lichess-org/chess-openings/master/{letter}.tsv"
    res = requests.get(url)
    lines = [l.split("\t") for l in res.text.strip().split("\n")[1:]]
    for parts in lines:
        if len(parts) >= 2:
            eco_dfs.append({"eco": parts[0].strip(), "name": parts[1].strip()})

eco_df = pd.DataFrame(eco_dfs).drop_duplicates(subset="eco", keep="first")
eco_map = dict(zip(eco_df["eco"], eco_df["name"]))

print(f"✅ {len(eco_map)} códigos ECO cargados")
print(list(eco_map.items())[:3])

✅ 498 códigos ECO cargados
[('A00', 'Amar Opening'), ('A01', 'Nimzo-Larsen Attack'), ('A02', 'Bird Opening')]


In [11]:
def parse_opening(name):
    if not name or name == "?":
        return "Unknown", "Unknown", "Unknown"

    if ":" in name:
        parts = name.split(":", 1)
        family = parts[0].strip()
        rest = parts[1].strip()
        if "," in rest:
            sub_parts = rest.split(",", 1)
            variation = sub_parts[0].strip()
            subvariation = sub_parts[1].strip()
        else:
            variation = rest
            subvariation = rest
    else:
        family = name.strip()
        variation = name.strip()
        subvariation = name.strip()

    return family, variation, subvariation


rows = []
for game in tqdm(games):
    eco = game.headers.get("ECO", "?")
    result = game.headers.get("Result", "?")
    is_white = game.headers.get("White", "").lower() == USERNAME.lower()
    time_control = game.headers.get("TimeControl", "?")

    # Resolver nombre desde ECO map
    opening_name = eco_map.get(eco, None)

    if result == "1-0":
        outcome = "win" if is_white else "loss"
    elif result == "0-1":
        outcome = "loss" if is_white else "win"
    elif result == "1/2-1/2":
        outcome = "draw"
    else:
        outcome = "unknown"

    family, variation, subvariation = parse_opening(opening_name)

    rows.append({
        "eco": eco,
        "opening_raw": opening_name,
        "family": family,
        "variation": variation,
        "subvariation": subvariation,
        "outcome": outcome,
        "color": "white" if is_white else "black",
        "time_control": time_control
    })

df_raw = pd.DataFrame(rows)
print(f"✅ {len(df_raw)} partidas procesadas")
print(f"Con apertura: {(df_raw['family'] != 'Unknown').sum()}")
print(f"Sin apertura: {(df_raw['family'] == 'Unknown').sum()}")
df_raw.head(10)

100%|██████████| 453/453 [00:00<00:00, 82854.51it/s]

✅ 453 partidas procesadas
Con apertura: 453
Sin apertura: 0


,eco,opening_raw,family,variation,subvariation,outcome,color,time_control
0,B22,Sicilian Defense: Alapin Variation,Sicilian Defense,Alapin Variation,Alapin Variation,loss,white,1800
1,B00,Barnes Defense,Barnes Defense,Barnes Defense,Barnes Defense,loss,white,600
2,A00,Amar Opening,Amar Opening,Amar Opening,Amar Opening,loss,black,600
3,B00,Barnes Defense,Barnes Defense,Barnes Defense,Barnes Defense,loss,white,1800
4,B07,Czech Defense,Czech Defense,Czech Defense,Czech Defense,loss,black,600
5,D02,London System: Poisoned Pawn Variation,London System,Poisoned Pawn Variation,Poisoned Pawn Variation,loss,white,1/604800
6,C26,Bishop's Opening: Horwitz Gambit,Bishop's Opening,Horwitz Gambit,Horwitz Gambit,loss,white,1/604800
7,D02,London System: Poisoned Pawn Variation,London System,Poisoned Pawn Variation,Poisoned Pawn Variation,loss,black,1/604800
8,C47,Four Knights Game,Four Knights Game,Four Knights Game,Four Knights Game,loss,black,1/604800
9,C50,Four Knights Game: Italian Variation,Four Knights Game,Italian Variation,Italian Variation,loss,black,1/604800


In [12]:
# Conteo por jerarquía para el sunburst
df_sunburst = (
    df_raw
    .groupby(["family", "variation", "subvariation"])
    .agg(games=("outcome", "count"),
         wins=("outcome", lambda x: (x == "win").sum()),
         losses=("outcome", lambda x: (x == "loss").sum()),
         draws=("outcome", lambda x: (x == "draw").sum()))
    .reset_index()
)

df_sunburst["win_rate"] = (df_sunburst["wins"] / df_sunburst["games"]).round(3)

print(f"✅ {len(df_sunburst)} combinaciones únicas de apertura")
df_sunburst.head(10)

✅ 63 combinaciones únicas de apertura


,family,variation,subvariation,games,wins,losses,draws,win_rate
0,Alekhine Defense,Alekhine Defense,Alekhine Defense,6,2,4,0,0.333
1,Amar Opening,Amar Opening,Amar Opening,19,11,8,0,0.579
2,Amazon Attack,Amazon Attack,Amazon Attack,13,6,6,1,0.462
3,Amazon Attack,Siberian Attack,Siberian Attack,1,1,0,0,1.000
4,Australian Defense,Australian Defense,Australian Defense,1,0,1,0,0.000
5,Barnes Defense,Barnes Defense,Barnes Defense,25,10,15,0,0.400
6,Barnes Opening,Walkerling,Walkerling,17,9,8,0,0.529
7,Bird Opening,Dutch Variation,Batavo Gambit,1,0,1,0,0.000
8,Bishop's Opening,Berlin Defense,Berlin Defense,29,13,15,1,0.448
9,Bishop's Opening,Bishop's Opening,Bishop's Opening,2,0,2,0,0.000


In [13]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# CSV completo con jerarquía (para sunburst en rawgraph)
sunburst_path = f"{OUTPUT_DIR}/openings_sunburst.csv"
df_sunburst.to_csv(sunburst_path, index=False)

# CSV con datos crudos por partida (útil para otros análisis)
raw_path = f"{OUTPUT_DIR}/openings_raw.csv"
df_raw.to_csv(raw_path, index=False)

print(sunburst_path)
print(raw_path)

/content/drive/MyDrive/InfoVis/tp-chess/chess_data/openings_sunburst.csv
/content/drive/MyDrive/InfoVis/tp-chess/chess_data/openings_raw.csv


In [19]:
MIN_GAMES = 25  # ajustá el umbral

top_families = df_sunburst.groupby("family")["games"].sum()
top_families = top_families[top_families >= MIN_GAMES].index

df_sunburst_top = df_sunburst[df_sunburst["family"].isin(top_families)]
print(f"{len(df_sunburst_top)} filas, {df_sunburst_top['family'].nunique()} familias")

top_path = f"{OUTPUT_DIR}/openings_sunburst_top.csv"
df_sunburst_top.to_csv(top_path, index=False)
print(top_path)

22 filas, 6 familias
/content/drive/MyDrive/InfoVis/tp-chess/chess_data/openings_sunburst_top.csv
